# PocketWise

In [1]:
import os
from getpass import getpass

os.environ["PW_HOST"] = "localhost"
os.environ["PW_PORT"] = "3306"
os.environ["PW_USER"] = "root"
os.environ["PW_PASSWORD"] = getpass("MySQL password: ")
print("MySQL connection settings loaded for this notebook session.")

MySQL password:  ········


MySQL connection settings loaded for this notebook session.


In [2]:
APP_SOURCE = 'import os\nimport csv\nimport io\nimport calendar\nfrom datetime import date, datetime\nfrom decimal import Decimal\n\nimport mysql.connector\nimport streamlit as st\nimport matplotlib.pyplot as plt\n\nst.set_page_config(page_title="PocketWise", page_icon="💚", layout="wide")\nst.markdown("""<style>\n    .block-container {padding-top:1.5rem;max-width:1180px}\n    h1,h2,h3 {letter-spacing:-.025em}\n    [data-testid="stMetric"] {background:#f4fbf7;border:1px solid #d9eee2;padding:18px;border-radius:14px}\n    div.stButton > button[kind="primary"] {background:#16845a;border-color:#16845a}\n    </style>""",unsafe_allow_html=True)\n\nCATEGORIES=["Food","Travel","Shopping","Education","Health","Entertainment","Bills","Other"]\n\ndef conn():\n    return mysql.connector.connect(host=os.getenv("PW_HOST","localhost"),port=int(os.getenv("PW_PORT","3306")),user=os.getenv("PW_USER","root"),password=os.getenv("PW_PASSWORD",""),database="pocketwise")\n\ndef setup_db():\n    c=mysql.connector.connect(host=os.getenv("PW_HOST","localhost"),port=int(os.getenv("PW_PORT","3306")),user=os.getenv("PW_USER","root"),password=os.getenv("PW_PASSWORD",""))\n    try:\n        cur=c.cursor()\n        cur.execute("CREATE DATABASE IF NOT EXISTS pocketwise CHARACTER SET utf8mb4")\n        c.commit()\n    finally: c.close()\n    with conn() as c:\n        cur=c.cursor()\n        cur.execute("""CREATE TABLE IF NOT EXISTS expenses (\n          id INT PRIMARY KEY AUTO_INCREMENT, day DATE NOT NULL,\n          category VARCHAR(60) NOT NULL, amount DECIMAL(12,2) NOT NULL,\n          note VARCHAR(500) NOT NULL DEFAULT \'\',\n          CHECK (amount > 0))""")\n        cur.execute("""CREATE TABLE IF NOT EXISTS budgets (\n          month CHAR(7) PRIMARY KEY, amount DECIMAL(12,2) NOT NULL,\n          CHECK (amount > 0))""")\n        cur.execute("CREATE INDEX IF NOT EXISTS idx_expenses_day ON expenses(day)") if False else None\n        c.commit()\n\ndef query(sql, params=()):\n    c=conn()\n    try:\n        cur=c.cursor(dictionary=True)\n        cur.execute(sql,params)\n        return cur.fetchall()\n    finally: c.close()\n\ndef execute(sql, params=()):\n    c=conn()\n    try:\n        cur=c.cursor()\n        cur.execute(sql,params)\n        c.commit()\n        return cur.lastrowid\n    except:\n        c.rollback()\n        raise\n    finally: c.close()\n\ndef month_bounds(month):\n    year,mo=map(int,month.split(\'-\'))\n    start=date(year,mo,1)\n    end=date(year+1,1,1) if mo==12 else date(year,mo+1,1)\n    return start,end\n\ndef rup(n): return \'₹\'+format(float(n or 0),\',.2f\')\n\ntry:\n    setup_db()\nexcept mysql.connector.Error as e:\n    st.error("Cannot connect to MySQL. Start MySQL Server, set notebook credentials, then restart the Streamlit app.")\n    st.code(str(e))\n    st.stop()\n\nst.sidebar.markdown("# 💚 PocketWise")\nst.sidebar.caption("Spend Smart. Save More.")\npage=st.sidebar.radio("Navigation",["Dashboard","Add Expense","Manage Expenses","Monthly Budget","Analytics"])\ntoday=date.today()\nmonth=st.sidebar.selectbox("Reporting month",[(today.replace(day=1)) if False else f\'{y:04d}-{m:02d}\' for y in range(today.year-1,today.year+2) for m in range(1,13)],index=(1*12+today.month-1))\nstart,end=month_bounds(month)\n\nif page=="Dashboard":\n    st.title("💚 PocketWise Dashboard")\n    st.caption(f"Overview for {month}")\n    stats=query("SELECT COALESCE(SUM(amount),0) AS spent,COUNT(*) AS n,COALESCE(AVG(amount),0) AS avg FROM expenses WHERE day >= %s AND day < %s",(start,end))[0]\n    rec=query("SELECT amount FROM budgets WHERE month=%s",(month,))\n    budget=rec[0][\'amount\'] if rec else Decimal(\'0\')\n    spent=stats[\'spent\']; remaining=budget-spent\n    a,b,c,d=st.columns(4)\n    a.metric("Total spent",rup(spent)); b.metric("Transactions",stats[\'n\']); c.metric("Monthly budget",rup(budget) if budget else "Not set"); d.metric("Remaining",rup(remaining) if budget else "—")\n    st.caption(f"Average transaction: {rup(stats[\'avg\'])}")\n    if budget:\n        fraction=float(spent/budget)\n        st.progress(min(1,max(0,fraction)),text=f"Budget used: {fraction*100:.1f}%")\n        if fraction>=1: st.error("Budget exceeded!")\n        elif fraction>=.8: st.warning("80% or more of the monthly budget has been used.")\n        else: st.success("Spending is currently below 80% of the budget.")\n    else: st.info("Set a monthly budget to enable alerts.")\n    days=calendar.monthrange(start.year,start.month)[1]\n    elapsed=(today.day if today.year==start.year and today.month==start.month else days if start<today else 0)\n    if elapsed:\n        forecast=float(spent)/elapsed*days\n        st.info(f"Simple projected month-end spending: **{rup(forecast)}** (average per elapsed calendar day)")\n    else: st.info("Projection is only shown for current or completed months.")\n    categories=query("SELECT category,SUM(amount) AS total FROM expenses WHERE day >= %s AND day < %s GROUP BY category ORDER BY total DESC",(start,end))\n    daily=query("SELECT day,SUM(amount) AS total FROM expenses WHERE day >= %s AND day < %s GROUP BY day ORDER BY day",(start,end))\n    left,right=st.columns(2)\n    if categories:\n        with left:\n            st.subheader("Spending by category")\n            fig,ax=plt.subplots(figsize=(5.8,3.8)); ax.pie([float(r[\'total\']) for r in categories],labels=[r[\'category\'] for r in categories],autopct=\'%1.0f%%\',startangle=90); fig.tight_layout();st.pyplot(fig);plt.close(fig)\n        with right:\n            st.subheader("Daily spending")\n            fig,ax=plt.subplots(figsize=(5.8,3.8));ax.bar([r[\'day\'].strftime(\'%d\') for r in daily],[float(r[\'total\']) for r in daily],color=\'#178a60\');ax.set(xlabel="Day of month",ylabel="₹ spent");fig.tight_layout();st.pyplot(fig);plt.close(fig)\n    else: st.info("No expenses for this month. Add an expense to see charts.")\n\nelif page=="Add Expense":\n    st.title("Add an Expense")\n    with st.form("add"):\n        d=st.date_input("Date",value=today)\n        category=st.selectbox("Category",CATEGORIES)\n        amount=st.number_input("Amount (₹)",min_value=0.01,value=100.0,step=10.0)\n        note=st.text_input("Description (optional)",max_chars=500)\n        ok=st.form_submit_button("Save Expense",type="primary")\n    if ok:\n        execute("INSERT INTO expenses(day,category,amount,note) VALUES (%s,%s,%s,%s)",(d,category,str(round(amount,2)),note))\n        st.success("Expense saved in MySQL.")\n\nelif page=="Manage Expenses":\n    st.title("Manage Expenses")\n    records=query("SELECT id,day,category,amount,note FROM expenses WHERE day >= %s AND day < %s ORDER BY day DESC,id DESC",(start,end))\n    st.caption(f"{len(records)} expenses for {month}")\n    if records:\n        st.dataframe([{**r,"amount":float(r[\'amount\'])} for r in records],use_container_width=True,hide_index=True)\n        sio=io.StringIO(); writer=csv.writer(sio);writer.writerow(["id","day","category","amount","note"])\n        for r in records: writer.writerow([r[\'id\'],r[\'day\'].isoformat(),r[\'category\'],r[\'amount\'],r[\'note\']])\n        st.download_button("Download CSV",data=sio.getvalue().encode(\'utf-8-sig\'),file_name=f\'pocketwise_{month}.csv\',mime=\'text/csv\')\n        options={f"#{r[\'id\']} — {r[\'day\']} — {r[\'category\']} — {rup(r[\'amount\'])}":r for r in records}\n        selected=options[st.selectbox("Choose a transaction to modify",list(options))]\n        with st.form(\'edit\'):\n            d=st.date_input("Date",value=selected[\'day\'])\n            cat=st.selectbox("Category",CATEGORIES,index=CATEGORIES.index(selected[\'category\']) if selected[\'category\'] in CATEGORIES else len(CATEGORIES)-1)\n            amt=st.number_input("Amount (₹)",min_value=0.01,value=float(selected[\'amount\']),step=10.0)\n            note=st.text_input("Description",value=selected[\'note\'],max_chars=500)\n            save=st.form_submit_button("Update selected expense")\n        if save:\n            execute("UPDATE expenses SET day=%s,category=%s,amount=%s,note=%s WHERE id=%s",(d,cat,str(round(amt,2)),note,selected[\'id\']))\n            st.success("Expense updated.");st.rerun()\n        st.divider()\n        confirm=st.checkbox("I understand this will permanently delete the selected transaction")\n        if st.button("Delete selected expense",disabled=not confirm):\n            execute("DELETE FROM expenses WHERE id=%s",(selected[\'id\'],))\n            st.success("Expense deleted.");st.rerun()\n    else: st.info("No expenses found for this month.")\n\nelif page=="Monthly Budget":\n    st.title("Monthly Budget")\n    budgetrow=query("SELECT amount FROM budgets WHERE month=%s",(month,))\n    current=float(budgetrow[0][\'amount\']) if budgetrow else 1000.0\n    with st.form(\'budget\'):\n        amt=st.number_input(f"Budget for {month} (₹)",min_value=1.0,value=current,step=100.0)\n        save=st.form_submit_button("Save Budget",type="primary")\n    if save:\n        execute("INSERT INTO budgets(month,amount) VALUES(%s,%s) ON DUPLICATE KEY UPDATE amount=VALUES(amount)",(month,str(round(amt,2))))\n        st.success("Monthly budget saved.");st.rerun()\n\nelif page=="Analytics":\n    st.title("Expense Analytics")\n    s=query("SELECT COUNT(*) AS n,COALESCE(SUM(amount),0) AS total,COALESCE(AVG(amount),0) AS avg,COALESCE(MAX(amount),0) AS largest FROM expenses WHERE day >= %s AND day < %s",(start,end))[0]\n    a,b,c=st.columns(3);a.metric("Transactions",s[\'n\']);b.metric("Average expense",rup(s[\'avg\']));c.metric("Largest expense",rup(s[\'largest\']))\n    rows=query("SELECT category,COUNT(*) AS transactions,SUM(amount) AS total,AVG(amount) AS average FROM expenses WHERE day >= %s AND day < %s GROUP BY category ORDER BY total DESC",(start,end))\n    if rows:\n        st.subheader("Category summary — computed by MySQL GROUP BY")\n        st.dataframe([{"category":r[\'category\'],"transactions":r[\'transactions\'],"total (₹)":float(r[\'total\']),"average (₹)":round(float(r[\'average\']),2)} for r in rows],hide_index=True,use_container_width=True)\n        st.success(f"Top spending category: {rows[0][\'category\']}")\n    else: st.info("No expense data in this month.")\n    st.subheader("Monthly trend")\n    trends=query("SELECT DATE_FORMAT(day,\'%%Y-%%m\') AS month,SUM(amount) AS total FROM expenses GROUP BY DATE_FORMAT(day,\'%%Y-%%m\') ORDER BY month")\n    if trends:\n        fig,ax=plt.subplots(figsize=(9,3));ax.plot([r[\'month\'] for r in trends],[float(r[\'total\']) for r in trends],marker=\'o\',color=\'#16845a\');ax.set(ylabel="₹ spent",xlabel="Month");fig.autofmt_xdate();fig.tight_layout();st.pyplot(fig);plt.close(fig)\n    else: st.caption("Add expenses across multiple months to see the trend.")\n\n\n'

In [3]:
from pathlib import Path
import subprocess, sys, time, os

Path("app.py").write_text(APP_SOURCE, encoding="utf-8")
try:
    if "pocketwise_process" in globals() and pocketwise_process.poll() is None:
        pocketwise_process.terminate()
except Exception:
    pass
pocketwise_log = open("pocketwise_streamlit.log", "w", encoding="utf-8")
pocketwise_process = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py", "--server.headless=true", "--server.port=8502"],
    stdout=pocketwise_log, stderr=subprocess.STDOUT, env=os.environ.copy()
)
print("Open: http://localhost:8502")
print("Streamlit process ID:",pocketwise_process.pid)
print("If the site does not open, inspect pocketwise_streamlit.log (or choose another port).")

Open: http://localhost:8502
Streamlit process ID: 576
If the site does not open, inspect pocketwise_streamlit.log (or choose another port).
